In [2]:
import pandas as pd
import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup, Comment



In [7]:
def get_stealth_driver():
    options = webdriver.ChromeOptions()
    # Hides the automation signature
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    # Optional: options.add_argument("--headless") # Run without a popup window
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # Overwrite the webdriver property in the browser itself
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

print("Environment Is Ready.")

Environment Is Ready.


In [3]:
driver = get_stealth_driver()
game_links = []

try:
    url = "https://www.basketball-reference.com/leagues/NBA_2026_games-november.html"
    driver.get(url)
    time.sleep(5)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find('table', {'id': 'schedule'})
    
    if table:
        rows = table.find_all('tr')
        for row in rows:
            cell = row.find('td', {'data-stat': 'box_score_text'})
            if cell and cell.find('a'):
                full_link = "https://www.basketball-reference.com" + cell.find('a')['href']
                game_links.append(full_link)
        
    print(f"Successfully found {len(game_links)} game links.")
finally:
    driver.quit()

Successfully found 219 game links.


In [4]:
driver = get_stealth_driver()
all_player_stats = []

try:
    for i, link in enumerate(game_links):
        print(f"[{i+1}/{len(game_links)}] Scraping: {link}")
        
        driver.get(link)
        driver.execute_script("window.scrollTo(0, 400);")
        time.sleep(9)  # wait for page to fully load
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Find all "basic" box score tables directly
        tables = soup.find_all('table', id=lambda x: x and 'game-basic' in x)
        
        if tables:
            for table in tables:
                team_id = table.get('id', '').split('-')[1]  # e.g. "LAL" from "box-LAL-game-basic"
                rows = table.find('tbody').find_all('tr')
                
                for row in rows:
                    if 'thead' in row.get('class', []): 
                        continue
                    p_cell = row.find('th', {'data-stat': 'player'})
                    if p_cell and not row.find('td', {'data-stat': 'reason'}):
                        try:
                            all_player_stats.append({
                                'Game_Date': link.split('/')[-1][:8],
                                'Team': team_id,
                                'Player': p_cell.text,
                                'MP': row.find('td', {'data-stat': 'mp'}).text,
                                'PTS': int(row.find('td', {'data-stat': 'pts'}).text),
                                'TRB': int(row.find('td', {'data-stat': 'trb'}).text),
                                'AST': int(row.find('td', {'data-stat': 'ast'}).text),
                                'FGA': int(row.find('td', {'data-stat': 'fga'}).text),
                                'FG%': row.find('td', {'data-stat': 'fg_pct'}).text,
                                'STL': int(row.find('td', {'data-stat': 'stl'}).text),
                                'TOV': int(row.find('td', {'data-stat': 'tov'}).text)
                            })
                        except Exception as e:
                            continue
            print(f"✅ Stats found for {link}")
        else:
            print(f"❌ No stats table found for {link}")
        
        # Save checkpoint every 10 games
        if (i + 1) % 10 == 0:
            pd.DataFrame(all_player_stats).to_csv("nba_checkpoint.csv", index=False)
            print("--- Checkpoint Saved ---")

finally:
    driver.quit()
    final_df = pd.DataFrame(all_player_stats)
    final_df.to_csv("nba_final_dataset.csv", index=False)
    print(f"Scraping Complete! {len(final_df)} rows saved to nba_final_dataset.csv")


[1/219] Scraping: https://www.basketball-reference.com/boxscores/202511010MIL.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202511010MIL.html
[2/219] Scraping: https://www.basketball-reference.com/boxscores/202511010CHO.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202511010CHO.html
[3/219] Scraping: https://www.basketball-reference.com/boxscores/202511010IND.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202511010IND.html
[4/219] Scraping: https://www.basketball-reference.com/boxscores/202511010WAS.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202511010WAS.html
[5/219] Scraping: https://www.basketball-reference.com/boxscores/202511010BOS.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202511010BOS.html
[6/219] Scraping: https://www.basketball-reference.com/boxscores/202511010DET.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202511010DET.html
[7/2

In [12]:
driver = get_stealth_driver()
december_game_links = []

try:
    url = "https://www.basketball-reference.com/leagues/NBA_2026_games-december.html"
    driver.get(url)
    time.sleep(5)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find('table', {'id': 'schedule'})
    
    if table:
        rows = table.find_all('tr')
        for row in rows:
            cell = row.find('td', {'data-stat': 'box_score_text'})
            if cell and cell.find('a'):
                full_link = "https://www.basketball-reference.com" + cell.find('a')['href']
                december_game_links.append(full_link)
        
    print(f"Successfully found {len(december_game_links)} game links.")
finally:
    driver.quit()


Successfully found 198 game links.


In [13]:
driver = get_stealth_driver()
all_player_stats = []

try:
    for i, link in enumerate(december_game_links):
        print(f"[{i+1}/{len(december_game_links)}] Scraping: {link}")
        
        driver.get(link)
        driver.execute_script("window.scrollTo(0, 400);")
        time.sleep(9)  # wait for page to fully load
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Find all "basic" box score tables directly
        tables = soup.find_all('table', id=lambda x: x and 'game-basic' in x)
        
        if tables:
            for table in tables:
                team_id = table.get('id', '').split('-')[1]  # e.g. "LAL" from "box-LAL-game-basic"
                rows = table.find('tbody').find_all('tr')
                
                for row in rows:
                    if 'thead' in row.get('class', []): 
                        continue
                    p_cell = row.find('th', {'data-stat': 'player'})
                    if p_cell and not row.find('td', {'data-stat': 'reason'}):
                        try:
                            all_player_stats.append({
                                'Game_Date': link.split('/')[-1][:8],
                                'Team': team_id,
                                'Player': p_cell.text,
                                'MP': row.find('td', {'data-stat': 'mp'}).text,
                                'PTS': int(row.find('td', {'data-stat': 'pts'}).text),
                                'TRB': int(row.find('td', {'data-stat': 'trb'}).text),
                                'AST': int(row.find('td', {'data-stat': 'ast'}).text),
                                'FGA': int(row.find('td', {'data-stat': 'fga'}).text),
                                'FG%': row.find('td', {'data-stat': 'fg_pct'}).text,
                                'STL': int(row.find('td', {'data-stat': 'stl'}).text),
                                'TOV': int(row.find('td', {'data-stat': 'tov'}).text)
                            })
                        except Exception:
                            continue
            print(f"✅ Stats found for {link}")
        else:
            print(f"❌ No stats table found for {link}")
        
        # Save checkpoint every 10 games
        if (i + 1) % 10 == 0:
            pd.DataFrame(all_player_stats).to_csv("nba_checkpoint_december.csv", index=False)
            print("--- Checkpoint Saved ---")

finally:
    driver.quit()
    final_df = pd.DataFrame(all_player_stats)
    final_df.to_csv("nba_final_dataset_december.csv", index=False)
    print(f"Scraping Complete! {len(final_df)} rows saved to nba_final_dataset_december.csv")


[1/198] Scraping: https://www.basketball-reference.com/boxscores/202512010DET.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202512010DET.html
[2/198] Scraping: https://www.basketball-reference.com/boxscores/202512010IND.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202512010IND.html
[3/198] Scraping: https://www.basketball-reference.com/boxscores/202512010WAS.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202512010WAS.html
[4/198] Scraping: https://www.basketball-reference.com/boxscores/202512010BRK.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202512010BRK.html
[5/198] Scraping: https://www.basketball-reference.com/boxscores/202512010MIA.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202512010MIA.html
[6/198] Scraping: https://www.basketball-reference.com/boxscores/202512010ORL.html
✅ Stats found for https://www.basketball-reference.com/boxscores/202512010ORL.html
[7/1

In [ ]:
# Rename the November dataset file
os.rename("nba_final_dataset.csv", "nba_final_dataset_november.csv")



In [3]:
# Load both datasets
november_df = pd.read_csv("nba_final_dataset_november.csv")
december_df = pd.read_csv("nba_final_dataset_december.csv")

# Concatenate them into one DataFrame
combined_df = pd.concat([november_df, december_df], ignore_index=True)

# Save the combined dataset
combined_df.to_csv("nba_final_dataset.csv", index=False)

print(f"Merge complete")


Merge complete
